In [1]:
%matplotlib qt5

import hyperspy.api as hs
import pyxem as pxm
import numpy as np
import orix
import matplotlib.pyplot as plt
import plotly.graph_objects as go
import matplotlib as mpl
import pickle

from pathlib import Path
from diffpy.structure import Atom, Lattice, Structure
from diffsims.generators.simulation_generator import SimulationGenerator
from orix.vector import Miller, Vector3d
from orix.plot import IPFColorKeyTSL
from matplotlib.lines import Line2D
from dataclasses import dataclass
from sklearn.decomposition import PCA

In [2]:
## Paths used for loading and saving data
path = Path("D:\\Datasets\\SPED_tilt_series\\SPED_tilt_series")
path_calibrated = Path("D:\\Datasets\\SPED_tilt_series\\series_calibrated")

path_target = Path("D:\\Datasets\\SPED_tilt_series\\target_areas")
path_reference = Path("D:\\Datasets\\SPED_tilt_series\\reference_areas")

path_target_masked = Path("D:\\Datasets\\SPED_tilt_series\\target_areas_masked")
path_reference_masked = Path("D:\\Datasets\\SPED_tilt_series\\reference_areas_masked")

target_files = sorted(path_target.glob("*.hspy"))
reference_files = sorted(path_reference.glob("*.hspy"))

target_files_masked = sorted(path_target_masked.glob("*.hspy"))
reference_files_masked = sorted(path_reference_masked.glob("*.hspy"))

## Preprocessing

In [ ]:
# The scaling factor used was determined by manually calibrating one of the datasets and extracting the scale factor from the calibrated data. #
def preprocess_data(path, path_calibrated, path_target, path_reference, scale_factor):
    files = sorted(path.glob("*.hspy"))
    for i in files:
        filename = i.stem
        number = filename.rsplit("_", 1)[-1]

        ## Calibrate the data ##
        load_i = hs.load(i)
        load_i.axes_manager[2].scale = scale_factor
        load_i.axes_manager[3].scale = scale_factor

        shifts = load_i.get_direct_beam_position(method="blur", sigma=5, half_square_width=10)
        linear_shifts = shifts.get_linear_plane()
        load_i_centered = load_i.center_direct_beam(shifts=linear_shifts, inplace=False)


        ## Select region of interest ##
        reference_rectangle = hs.roi.RectangularROI(left=0, top=0, right=10, bottom=10)
        target_rectangle = hs.roi.RectangularROI(left=0, top=0, right=10, bottom=10)

        load_i_centered.plot()

        reference_roi = reference_rectangle.interactive(load_i_centered)
        target_roi = target_rectangle.interactive(load_i_centered, color='C1')
        
        plt.show(block=True)

        reference_roi.save(path_reference / f"SPED_tilt_series_reference_{number}.hspy")
        target_roi.save(path_target / f"SPED_tilt_series_target_{number}.hspy")
        load_i_centered.save(path_calibrated / f"SPED_tilt_series_calibrated_{number}.hspy")

## Masking

In [ ]:
def masking_function(path_target, path_reference, path_target_masked, path_reference_masked):
    files_target = sorted(path_target.glob("*.hspy"))
    files_reference = sorted(path_reference.glob("*.hspy"))

    for number, j in enumerate(zip(files_target, files_reference)):
        
        load_target = hs.load(j[0])
        load_reference = hs.load(j[1])

        peaks_target = load_target.get_diffraction_vectors(min_distance=18, threshold_abs=0.1)
        peaks_reference = load_reference.get_diffraction_vectors(min_distance=18, threshold_abs=0.1)

        mask_target = peaks_target.to_mask(disk_r = 7)
        mask_reference = peaks_reference.to_mask(disk_r = 7)

        masked_target = load_target * mask_target
        masked_reference = load_reference * mask_reference

        masked_target.save(path_target_masked / f"target_masked_{number}.hspy")
        masked_reference.save(path_reference_masked / f"reference_masked_{number}.hspy")
    
masking_function(path_target, path_reference, path_target_masked, path_reference_masked)

## Simulation

In [3]:
# Define the crystal structure for aluminum (Al)
a = 4.05
atoms = [Atom('Al', [0, 0, 0]), Atom('Al', [0.5, 0.5, 0]), Atom('Al', [0.5, 0, 0.5]), Atom('Al', [0, 0.5, 0.5])]
lattice = Lattice(a, a, a, 90, 90, 90)
phase = orix.crystal_map.Phase(name='Al', space_group=225, structure=Structure(atoms, lattice))

angular_resolution = 0.5
grid = orix.sampling.get_sample_reduced_fundamental(angular_resolution, point_group=phase.point_group)
orientations = orix.quaternion.Orientation(grid, symmetry=phase.point_group)

simgen = SimulationGenerator(precession_angle=1., minimum_intensity=1e-5)
simulations = simgen.calculate_diffraction2d(phase=phase, rotation=grid, reciprocal_radius=1.5, with_direct_beam=False, max_excitation_error=0.01)

# Create IPF color keys for x, y, and z directions
key_x = IPFColorKeyTSL(phase.point_group, Vector3d.xvector())
key_y = IPFColorKeyTSL(phase.point_group, Vector3d.yvector())
key_z = IPFColorKeyTSL(phase.point_group, Vector3d.zvector())

In [4]:
@dataclass
class SPEDResult:
    results: object # Indices, correlations, orientations
    orientations: object   
    
    def ipf_correlation(self):        
        fig = plt.figure(figsize=(12, 6))

        ax = fig.add_subplot(121, projection="ipf", symmetry=phase.point_group)
        ax_dp = fig.add_subplot(122)

        ax.scatter(self.orientations, c=self.results[:, 1], cmap='magma')
        ax.scatter(self.orientations[0], c="r", marker="x")
        
        first = self.results[:, 0].astype('int16')[0]
        
        sim_i = simulations.irot[first]
        pat = sim_i.get_diffraction_pattern(
            shape=(256, 256),        # pick your detector size
            sigma=2,                 # spot size in pixels
            calibration=0.01,        # real->pixel scale (tune to match your data)
            direct_beam_position=(128, 128),
            normalize=True,
        )

        ax_dp.imshow(pat, vmax=1, cmap='magma')
        ax_dp.set_title(f"Simulated DP")
        ax_dp.axis('off')

        plt.tight_layout()
        plt.show()

In [5]:
def template(target_data, ref_data):
    ref_data.change_dtype("float32")
    target_data.change_dtype("float32")

    ny, nx = target_data.axes_manager.signal_shape
    center = (ny/2 - 0.5, nx/2 - 0.5)
    
    ref_data.calibration(center=center)
    target_data.calibration(center=center)
    
    target_azi = target_data.get_azimuthal_integral2d(npt=256)
    ref_azi = ref_data.get_azimuthal_integral2d(npt=256)
    
    target_results = target_azi.get_orientation(simulations, n_best=grid.size, frac_keep=0.5)
    ref_results = ref_azi.get_orientation(simulations, n_best=grid.size, frac_keep=0.5)
    
    target_orientations = target_results.to_single_phase_orientations()
    reference_orientations = ref_results.to_single_phase_orientations()
    
    target = SPEDResult(
        results=target_results.data,
        orientations=target_orientations)
    
    reference = SPEDResult(
        results=ref_results.data,  
        orientations=reference_orientations)
    
    return target, reference

In [ ]:
def tilt_results():
    misorientation_array = np.empty((len(target_files_masked), 2), dtype=object)

    for i, files in enumerate(zip(target_files_masked, reference_files_masked)):
        
        target_data = hs.load(files[0])
        reference_data = hs.load(files[1])

        target_sum = target_data.sum() ** 0.5 - 5
        reference_mean = reference_data.sum() ** 0.5 - 5

        target_result, reference_result = template(target_sum, reference_mean)

        misorientation_array[i,0] = target_result 
        misorientation_array[i,1] = reference_result
    
    return misorientation_array

result_array = tilt_results()
pickle.dump(result_array, open("result_array.pkl", "wb"))

In [6]:
path = Path("C:\\Users\\denis\\Documents\\GitHub\\Masteroppgave\\Results") / "result_array.pkl"

with open(path, "rb") as f:
    result_array = pickle.load(f)

In [106]:
def return_orix_orientation(result_array):
    orientation_target = np.empty(60, dtype=object)
    orientation_reference = np.empty(60, dtype=object)

    for number, i in enumerate(result_array):
        orientation_target[number] = i[0].orientations
        orientation_reference[number] = i[1].orientations

    quat_array = np.stack([j.data for j in orientation_target])
    orientation_target = orix.quaternion.Orientation(quat_array, symmetry=phase.point_group)

    quat_array = np.stack([k.data for k in orientation_reference])
    orientation_reference = orix.quaternion.Orientation(quat_array, symmetry=phase.point_group)

    return orientation_target, orientation_reference

orientation_target, orientation_reference = return_orix_orientation(result_array)
axis_angle_target = orientation_target.to_axes_angles()
axis_angle_reference = orientation_reference.to_axes_angles()

In [89]:
def plot_ipf(orientation, line=None):
    c_scalar = np.arange(1, 61)

    fig = plt.figure()
    ax = fig.add_subplot(projection='ipf', symmetry=phase.point_group)
    if line is not None:
        ax.scatter(line, c='k', linewidth=0.4)
    ax.scatter(orientation, c=c_scalar)
    

    # Create colorbar manually
    norm = mpl.colors.Normalize(vmin=c_scalar.min(), vmax=c_scalar.max())
    sm = mpl.cm.ScalarMappable(norm=norm)
    sm.set_array([])

    cbar = plt.colorbar(sm, ax=ax)
    cbar.set_label("Dataset number")
    plt.show()

#plot_ipf(orientation_target[:, 0])
#plot_ipf(orientation_reference[:, 0])

In [100]:
axis_data_target = np.array(axis_angle_target.xyz).transpose()
axis_data_reference = np.array(axis_angle_reference.xyz).transpose()

def fit_line_pca(points):
    centroid = points.mean(axis=0)
    pca = PCA(n_components=1)
    pca.fit(points)
    direction = pca.components_[0]
    direction = direction / np.linalg.norm(direction)
    return centroid, direction

def point_line_dist(points, point_on_line, direction):
    diff = points - point_on_line
    return np.linalg.norm(np.cross(diff, direction), axis=1)

def ransac_line_fit(points, n_iter=2000, threshold=0.05, min_inliers=10):
    n = len(points)
    best_inliers = None
    best_count = 0
    best_model = None

    for _ in range(n_iter):
        idx = np.random.choice(n, 2, replace=False)
        p1, p2 = points[idx]

        direction = p2 - p1
        norm = np.linalg.norm(direction)
        if norm < 1e-12:
            continue
        direction = direction / norm

        dist = point_line_dist(points, p1, direction)
        inliers = dist < threshold
        count = np.sum(inliers)

        if count > best_count:
            best_count = count
            best_inliers = inliers
            best_model = (p1, direction)

    if best_inliers is None or best_count < min_inliers:
        raise ValueError("No good line found. Try increasing threshold or n_iter.")

    inlier_points = points[best_inliers]
    centroid, direction = fit_line_pca(inlier_points)

    return centroid, direction, inlier_points, best_inliers

centroid_target, direction_target, inliers_target, inlier_mask_target = ransac_line_fit(
    axis_data_target[0, :, :],
    n_iter=3000,
    threshold=0.1,
    min_inliers=10
)

centroid_reference, direction_reference, inliers_reference, inlier_mask_reference = ransac_line_fit(
    axis_data_reference[0, :, :],
    n_iter=3000,
    threshold=0.1,
    min_inliers=10
)

In [101]:
def expected_t_from_local_steps(inline_idx, line_points, query_idx):

    inline_idx = np.asarray(inline_idx)
    line_points = np.asarray(line_points)
    query_idx = np.asarray(query_idx)

    if len(inline_idx) < 2:
        raise ValueError("Need at least two inline points.")

    idx_diff = np.diff(inline_idx)
    line_diff = np.diff(line_points)
    steps = line_diff / idx_diff   # local step per index in each interval

    t_expected = np.empty(len(query_idx), dtype=float)

    for n, q in enumerate(query_idx):
        pos = np.searchsorted(inline_idx, q)

        # Case 1: q is before the first inline point
        if pos == 0:
            step = steps[0]
            t_expected[n] = line_points[0] + step * (q - inline_idx[0])

        # Case 2: q is after the last inline point
        elif pos == len(inline_idx):
            step = steps[-1]
            t_expected[n] = line_points[-1] + step * (q - inline_idx[-1])

        # Case 3: q lands exactly on an inline point
        elif inline_idx[pos] == q:
            t_expected[n] = line_points[pos]

        # Case 4: q lies between two inline points
        else:
            left_idx = inline_idx[pos - 1]
            t_left = line_points[pos - 1]
            step = steps[pos - 1]
            t_expected[n] = t_left + step * (q - left_idx)

    return t_expected

In [126]:
nonl_target = np.transpose(axis_data_target[:,~inlier_mask_target,:], (1, 0, 2))
nonl_reference = np.transpose(axis_data_reference[:,~inlier_mask_reference,:], (1, 0, 2))

def choose_candidates_smooth(
    candidates,
    centroid,
    direction,
    inliers,
    inlier_mask,
    w_perp=2.0,
    w_t=0.8,
):
    direction = np.asarray(direction, dtype=float)
    direction = direction / np.linalg.norm(direction)
    centroid = np.asarray(centroid, dtype=float)

    N, K, _ = candidates.shape
    inlier_idx = np.where(inlier_mask)[0]

    if len(inlier_idx) < 2:
        raise ValueError("Need at least two inliers to estimate expected positions.")

    # t-values of trusted inlier points
    line_points = np.dot(inliers - centroid, direction)

    # expected t for every sequence index
    all_idx = np.arange(N)
    t_expected = expected_t_from_local_steps(inlier_idx, line_points, all_idx)

    # candidate coordinates relative to line
    diff = candidates - centroid[None, None, :]
    tvals = np.sum(diff * direction[None, None, :], axis=2)

    # perpendicular component
    perp_vec = diff - tvals[:, :, None] * direction[None, None, :]
    perp_dist = np.linalg.norm(perp_vec, axis=2)

    # along-line mismatch
    t_error = np.abs(tvals - t_expected[:, None])

    # combined score
    score = w_perp * perp_dist + w_t * t_error

    chosen_idx = np.argmin(score, axis=1)
    chosen_points = candidates[np.arange(N), chosen_idx]
    chosen_dist = perp_dist[np.arange(N), chosen_idx]

    return chosen_points, chosen_idx, chosen_dist, t_expected

chosen_points_target, chosen_idx_target, chosen_dist_target, t_expected_target = choose_candidates_smooth(  
    nonl_target, centroid_target, direction_target, inliers_target, inlier_mask_target)

chosen_points_reference, chosen_idx_reference, chosen_dist_reference, t_expected_reference = choose_candidates_smooth(
    nonl_reference, centroid_reference, direction_reference, inliers_reference, inlier_mask_reference)


In [127]:
def plot_line_fit(data, inliers, fixed_points, centroid, direction):
    fig = plt.figure(figsize=(8, 6))
    ax = fig.add_subplot(111, projection='3d')
    
    # --- all points ---
    ax.scatter(
        data[:, 0], data[:, 1], data[:, 2],
        alpha=0.2, s=10, label="All points"
    )
    
    # --- inliers ---
    ax.scatter(
        inliers[:, 0], inliers[:, 1], inliers[:, 2],
        s=20, label="Inliers"
    )
    
    # --- fixed points ---
    ax.scatter(
        fixed_points[:, 0], fixed_points[:, 1], fixed_points[:, 2],
        s=20, label="Fixed points"
    )
    
    # --- fitted line ---
    t = np.linspace(-1, 1, 60)
    line = centroid + t[:, None] * direction
    
    ax.plot(
        line[:, 0], line[:, 1], line[:, 2],
        linewidth=3, label="Fitted line"
    )
    
    ax.set_xlabel("x")
    ax.set_ylabel("y")
    ax.set_zlabel("z")
    
    ax.legend()
    plt.tight_layout()
    plt.show()

    return line

line_target = plot_line_fit(axis_data_target[0, :, :], inliers_target, chosen_points_target, centroid_target, direction_target)
line_reference = plot_line_fit(axis_data_reference[0, :, :], inliers_reference, chosen_points_reference, centroid_reference, direction_reference)

In [128]:
def axis_angle_array_to_quaternion(R):
    theta = np.linalg.norm(R, axis=1)
    
    q = np.zeros((len(R), 4))
    
    small = theta < 1e-12
    large = ~small
    
    # small angles
    q[small, 0] = 1.0
    
    # normal case
    n = R[large] / theta[large][:, None]
    
    q[large, 0] = np.cos(theta[large] / 2)
    q[large, 1:] = n * np.sin(theta[large] / 2)[:, None]
    
    return q


orix_line_target = orix.quaternion.Orientation(axis_angle_array_to_quaternion(line_target), symmetry=phase.point_group)
orix_line_reference = orix.quaternion.Orientation(axis_angle_array_to_quaternion(line_reference), symmetry=phase.point_group)

plot_ipf(orientation_target[:, 0], orix_line_target)
plot_ipf(orientation_reference[:, 0], orix_line_reference)

In [131]:
fixed_orientations_target = orientation_target
fixed_orientations_reference = orientation_reference

fixed_orientations_target[~inlier_mask_target, 0] = orientation_target[~inlier_mask_target, chosen_idx_target]
fixed_orientations_reference[~inlier_mask_reference, 0] = orientation_reference[~inlier_mask_reference, chosen_idx_reference]

plot_ipf(fixed_orientations_target[:, 0], orix_line_target)
plot_ipf(fixed_orientations_reference[:, 0], orix_line_reference)

In [ ]:
misorientation = fixed_orientations_target[:, 0].angle_with(fixed_orientations_reference[:, 0])
mean_misorientation = np.mean(misorientation)

number = np.arange(1, 61)
fig = go.Figure()
fig.add_trace(go.Scatter(x=number, y=misorientation, mode='markers+lines', name='Misorientation'))
fig.add_trace(go.Scatter(x=number, y=[mean_misorientation]*len(number), mode='lines', name='Mean Misorientation', line=dict(dash='dash')))
fig.update_layout(title='Misorientation vs Dataset Number', xaxis_title='Dataset Number', yaxis_title='Misorientation (degrees)')
fig.show()

In [ ]:
def axis_angle_plot(data_list, name):
    fig = go.Figure()
    
    rxyz = np.asarray(data_list.to_axes_angles().xyz)
    rx = rxyz[0].ravel()
    ry = rxyz[1].ravel()
    rz = rxyz[2].ravel()
    
    c_scalar = np.arange(len(rx))

    fig.add_trace(
    go.Scatter3d(
        x=rx, y=ry, z=rz,
        mode="markers+lines",
        marker=dict(
            color=c_scalar,              # 👈 THIS is the key
            colorscale="Viridis",        # choose colormap
            opacity=0.85,
            size=4,
            showscale=True,
            colorbar=dict(
                title="Datapoint index",
                x=-0.15,
                thickness=15,
                len=0.6
                )
            ),
        showlegend=True
        )
    )

    R = 3.4

    fig.update_layout(
        scene=dict(
            xaxis=dict(range=[-R, R], title="x"),
            yaxis=dict(range=[-R, R], title="y"),
            zaxis=dict(range=[-R, R], title="z"),
            aspectmode="cube"   # critical: equal scaling
        )
    )
    axis_lines = [
        # X-axis
        go.Scatter3d(
            x=[-R, R], y=[0, 0], z=[0, 0],
            mode="lines",
            line=dict(color="black", width=4),
            name="X axis",
            showlegend=False
        ),
        # Y-axis
        go.Scatter3d(
            x=[0, 0], y=[-R, R], z=[0, 0],
            mode="lines",
            line=dict(color="black", width=4),
            name="Y axis",
            showlegend=False
        ),
        # Z-axis
        go.Scatter3d(
            x=[0, 0], y=[0, 0], z=[-R, R],
            mode="lines",
            line=dict(color="black", width=4),
            name="Z axis",
            showlegend=False
        ),
    ]

    for trace in axis_lines:
        fig.add_trace(trace)

    fig.write_html(
    name,
    include_plotlyjs="cdn"
    )

axis_angle_plot(axis_angle_target, "axis_angle_target.html")
axis_angle_plot(axis_angle_reference, "axis_angle_reference.html")